**`Look at the class distribution — how many late, how many on time`**

In [3]:
import pandas as pd
orders_ml = pd.read_csv("artifacts/orders_ml.csv")
print("orders_ml loaded successfully.")
print("Shape:", orders_ml.shape)

orders_ml loaded successfully.
Shape: (99441, 21)


In [4]:
print(orders_ml.columns.tolist())

['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'item_count', 'unique_products', 'unique_sellers', 'total_price', 'total_freight', 'avg_item_price', 'payment_count', 'total_payment_value', 'max_installments', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']


In [5]:
orders_ml["order_delivered_customer_date"] = pd.to_datetime(
    orders_ml["order_delivered_customer_date"],errors="coerce")
    
orders_ml["order_estimated_delivery_date"] = pd.to_datetime(
    orders_ml["order_estimated_delivery_date"],errors="coerce")

In [7]:
orders_ml["is_late"] = (
    orders_ml["order_delivered_customer_date"]
    > orders_ml["order_estimated_delivery_date"]).astype(int)

In [8]:
valid_label_mask = (
    orders_ml["order_delivered_customer_date"].notna()
    & orders_ml["order_estimated_delivery_date"].notna())

print("Total orders:", len(orders_ml))
print("Orders with valid delivery dates:", valid_label_mask.sum())
print("Orders without valid delivery dates:", (~valid_label_mask).sum())

Total orders: 99441
Orders with valid delivery dates: 96476
Orders without valid delivery dates: 2965


In [11]:
labeled_orders = orders_ml[valid_label_mask].copy()
print("Labeled orders:", len(labeled_orders))

Labeled orders: 96476


In [12]:
label_counts = labeled_orders["is_late"].value_counts()
print(label_counts)

is_late
0    88649
1     7827
Name: count, dtype: int64


In [13]:
print(
    labeled_orders["is_late"]
    .value_counts()
    .rename(index={0: "On time", 1: "Late"}))

is_late
On time    88649
Late        7827
Name: count, dtype: int64


In [14]:
label_percentages = (
    labeled_orders["is_late"]
    .value_counts(normalize=True)
    .mul(100))

print(label_percentages)

is_late
0    91.887101
1     8.112899
Name: proportion, dtype: float64


`Only orders with valid actual and estimated delivery dates were retained for label creation. Orders missing either date were excluded because a reliable late/on-time label could not be determined.`

In [ ]:
# Class imbalance
class_counts = labeled_orders["is_late"].value_counts()
on_time_count = class_counts.get(0, 0)
late_count = class_counts.get(1, 0)
print("On-time orders:", on_time_count)
print("Late orders:", late_count)
if late_count > 0:
    print("On-time / Late ratio:", round(on_time_count / late_count, 2))

On-time orders: 88649
Late orders: 7827
On-time / Late ratio: 11.33


In [15]:
labeled_orders.to_csv(
    "artifacts/orders_labeled.csv",
    index=False)
print("Labeled dataset saved successfully.")

Labeled dataset saved successfully.
